In [39]:
import requests
import pandas as pd
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

In [38]:

def make_session():
    session = requests.Session()
    retry = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    return session

def fetch_smard(filter_id, region="DE", start_year=2022):
    session = make_session()
    base = f"https://www.smard.de/app/chart_data/{filter_id}/{region}"
    index_url = f"{base}/index_hour.json"

    response = session.get(index_url, headers=HEADERS, timeout=30)
    print(f"  Status: {response.status_code}")

    if response.status_code != 200 or not response.text.strip():
        print("  Failed: empty or bad response")
        return pd.DataFrame()

    idx = response.json()

    # filter timestamps to start_year onwards to reduce request volume
    cutoff_ms = pd.Timestamp(f"{start_year}-01-01").timestamp() * 1000
    timestamps = [ts for ts in idx["timestamps"] if ts >= cutoff_ms]
    print(f"  Fetching {len(timestamps)} chunks from {start_year} onwards")

    frames = []
    for i, ts in enumerate(timestamps):
        data_url = f"{base}/{filter_id}_{region}_hour_{ts}.json"
        try:
            r = session.get(data_url, headers=HEADERS, timeout=30)
            if r.status_code != 200 or not r.text.strip():
                continue
            data = r.json()
            if "series" in data:
                frames.append(pd.DataFrame(data["series"], columns=["timestamp", "value_mwh"]))
            if i % 10 == 0:
                print(f"  Chunk {i+1}/{len(timestamps)} done")
        except requests.exceptions.ReadTimeout:
            print(f"  Timeout on chunk {i}, skipping")
            time.sleep(2)
            continue
        time.sleep(0.3)

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames)
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    return df.set_index("timestamp").sort_index()

In [5]:
filter_ids = {
    "wind_onshore": 1223,
    "wind_offshore": 1224,
    "solar": 4067,
    "biomass": 1226,
    "hydro": 1225,
    "hard_coal": 1228,
    "lignite": 1229,
    "gas": 4169,
    "total_gen": 410
}

raw = {}
for source, fid in filter_ids.items():
    print(f"Fetching {source}...")
    df = fetch_smard(fid)
    if not df.empty:
        raw[source] = df
        print(f"  Successfully fetched {len(df)} records for {source}")
    else:
        print(f"  Failed to fetch data for {source}")
    time.sleep(1)  # Add delay between different sources

print(f"\nSuccessfully fetched data for {len(raw)} out of {len(filter_ids)} sources")
print("Available data sources:", list(raw.keys()))

Fetching wind_onshore...
  Status: 200
  Fetching 227 chunks from 2022 onwards
  Chunk 1/227 done
  Chunk 11/227 done
  Chunk 21/227 done
  Chunk 31/227 done
  Chunk 41/227 done
  Chunk 51/227 done
  Chunk 61/227 done
  Chunk 71/227 done
  Chunk 81/227 done
  Chunk 91/227 done
  Chunk 101/227 done
  Chunk 111/227 done
  Chunk 121/227 done
  Chunk 131/227 done
  Chunk 141/227 done
  Chunk 151/227 done
  Chunk 161/227 done
  Chunk 171/227 done
  Chunk 181/227 done
  Chunk 191/227 done
  Chunk 201/227 done
  Chunk 211/227 done
  Chunk 221/227 done
  Successfully fetched 38135 records for wind_onshore
Fetching wind_offshore...
  Status: 200
  Fetching 109 chunks from 2022 onwards
  Chunk 1/109 done
  Chunk 11/109 done
  Chunk 21/109 done
  Chunk 31/109 done
  Chunk 41/109 done
  Chunk 51/109 done
  Chunk 61/109 done
  Chunk 71/109 done
  Chunk 81/109 done
  Chunk 91/109 done
  Chunk 101/109 done
  Successfully fetched 18312 records for wind_offshore
Fetching solar...
  Status: 200
  Fetchi

In [46]:
# fetch nuclear and pumped storage separately
missing = {
    "nuclear": 1231,
    "pumped_storage": 1227,
    "other": 1228  
}

for source, fid in missing.items():
    print(f"Fetching {source}...")
    result = fetch_smard(fid, start_year=2022)
    if not result.empty:
        raw[source] = result
        print(f"  {len(result)} rows fetched")
    else:
        print(f"  Failed: {source}")

Fetching nuclear...
  Status: 404
  Failed: empty or bad response
  Failed: nuclear
Fetching pumped_storage...
  Status: 200
  Fetching 227 chunks from 2022 onwards
  Chunk 1/227 done
  Chunk 11/227 done
  Chunk 21/227 done
  Chunk 31/227 done
  Chunk 41/227 done
  Chunk 51/227 done
  Chunk 61/227 done
  Chunk 71/227 done
  Chunk 81/227 done
  Chunk 91/227 done
  Chunk 101/227 done
  Chunk 111/227 done
  Chunk 121/227 done
  Chunk 131/227 done
  Chunk 141/227 done
  Chunk 151/227 done
  Chunk 161/227 done
  Chunk 171/227 done
  Chunk 181/227 done
  Chunk 191/227 done
  Chunk 201/227 done
  Chunk 211/227 done
  Chunk 221/227 done
  38135 rows fetched
Fetching other...
  Status: 200
  Fetching 227 chunks from 2022 onwards
  Chunk 1/227 done
  Chunk 11/227 done
  Chunk 21/227 done
  Chunk 31/227 done
  Chunk 41/227 done
  Chunk 51/227 done
  Chunk 61/227 done
  Chunk 71/227 done
  Chunk 81/227 done
  Chunk 91/227 done
  Chunk 101/227 done
  Chunk 111/227 done
  Chunk 121/227 done
  Chunk 

In [47]:
frames = {}
for source, source_df in raw.items():
    numeric_cols = source_df.select_dtypes(include="number").columns
    if len(numeric_cols) == 0:
        continue
    frames[source] = source_df[numeric_cols[0]]

df = pd.DataFrame(frames).sort_index().fillna(0)
df = df[df["total_gen"] > 0]

# derive lignite only after accounting for all known sources
all_known = ["wind_onshore", "wind_offshore", "solar", "biomass",
             "hydro", "hard_coal", "gas", "nuclear", "pumped_storage"]
available_known = [c for c in all_known if c in df.columns]

df["lignite"] = (df["total_gen"] - df[available_known].sum(axis=1)).clip(lower=0, upper=18000)

# recalculate aggregates
renewable_cols = ["wind_onshore", "wind_offshore", "solar", "biomass", "hydro"]
fossil_cols = [c for c in ["hard_coal", "lignite", "gas"] if c in df.columns]

df["renewable_mwh"] = df[renewable_cols].sum(axis=1)
df["fossil_mwh"] = df[fossil_cols].sum(axis=1)
df["renewable_share"] = (df["renewable_mwh"] / df["total_gen"]).clip(upper=1.0)
df["fossil_share"] = (df["fossil_mwh"] / df["total_gen"]).clip(upper=1.0)

df["hour"] = df.index.hour
df["month"] = df.index.month
df["year"] = df.index.year
df["season"] = df["month"].map({
    12:"winter", 1:"winter", 2:"winter",
    3:"spring",  4:"spring",  5:"spring",
    6:"summer",  7:"summer",  8:"summer",
    9:"autumn",  10:"autumn", 11:"autumn"
})

print(df.columns.tolist())
print(df[["renewable_share", "fossil_share", "lignite"]].describe())
print(df["lignite"].quantile([0.1, 0.5, 0.9]))

['wind_onshore', 'wind_offshore', 'solar', 'biomass', 'hydro', 'hard_coal', 'gas', 'total_gen', 'pumped_storage', 'other', 'lignite', 'renewable_mwh', 'fossil_mwh', 'renewable_share', 'fossil_share', 'hour', 'month', 'year', 'season']
       renewable_share  fossil_share       lignite
count     37979.000000  37979.000000  37979.000000
mean          0.514753      0.303293  15777.300833
std           0.190865      0.094461   4454.538190
min           0.077784      0.001392      0.000000
25%           0.368840      0.270754  16004.770000
50%           0.491788      0.310294  18000.000000
75%           0.641476      0.361406  18000.000000
max           1.000000      0.589171  18000.000000
0.1     8754.828
0.5    18000.000
0.9    18000.000
Name: lignite, dtype: float64


In [45]:
df.to_csv("energiewende_tracker.csv")
print("Saved: energiewende_tracker.csv")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")

Saved: energiewende_tracker.csv
Rows: 37979 | Columns: 17


In [43]:
df.describe()

c:\Users\USER\OneDrive\Desktop\Energiewende-Tracker\Energiewende-Tracker---Germany\.venv\Lib\site-packages\pandas\core\nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,wind_onshore,wind_offshore,solar,biomass,hydro,hard_coal,gas,total_gen,lignite,renewable_mwh,fossil_mwh,renewable_share,fossil_share,hour,month,year
count,38135.000000,38135.000000,38135.000000,38135.000000,38135.000000,38135.000000,38135.000000,38135.000000,38135.000000,38135.000000,38135.000000,37979.000000,3.799100e+04,38135.000000,38135.000000,38135.000000
mean,9020.398903,1033.302904,12585.544955,1663.983134,2925.528318,125.916207,121.899949,53431.999280,17569.812572,27228.758213,17817.628728,0.515220,inf,11.499725,6.216415,2023.706569
std,3557.647390,1636.075343,9881.611479,410.246356,2000.586332,31.228202,103.050420,9952.968525,4797.456898,10648.643566,4823.255598,0.192147,NaN,6.922159,3.483807,1.270559
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-500.000000,0.000000,0.000000,0.000000,0.000000,0.077784,1.539346e-03,0.000000,1.000000,2022.000000
25%,6016.500000,0.000000,4705.250000,1353.250000,1129.250000,106.750000,72.840000,45877.980000,17541.085000,19512.850000,17783.220000,0.368840,3.010501e-01,5.500000,3.000000,2023.000000
50%,9374.360000,0.000000,9895.250000,1636.210000,2679.470000,117.240000,99.680000,53633.750000,20000.000000,25549.500000,20197.450000,0.491788,3.444727e-01,11.000000,6.000000,2024.000000
75%,11922.875000,2655.375000,18269.500000,1952.250000,4590.625000,138.000000,138.875000,60636.750000,20000.000000,34234.875000,20254.970000,0.641476,4.007261e-01,17.000000,9.000000,2025.000000
max,17173.250000,4117.500000,48499.500000,3078.250000,8448.340000,225.750000,936.280000,78680.500000,20000.000000,61603.000000,21070.780000,1.212678,inf,23.000000,12.000000,2026.000000


In [48]:
# test nuclear filter IDs
for fid in [1231, 1232, 4169, 126]:
    session = make_session()
    url = f"https://www.smard.de/app/chart_data/{fid}/DE/index_hour.json"
    r = session.get(url, headers=HEADERS, timeout=30)
    print(f"Filter {fid}: status {r.status_code}")

Filter 1231: status 404
Filter 1232: status 404
Filter 4169: status 200
Filter 126: status 200


In [49]:
test_126 = fetch_smard(126, start_year=2022)
print(test_126.shape)
print(test_126.describe())
print(test_126.head())

  Status: 200
  Fetching 227 chunks from 2022 onwards
  Chunk 1/227 done
  Chunk 11/227 done
  Chunk 21/227 done
  Chunk 31/227 done
  Chunk 41/227 done
  Chunk 51/227 done
  Chunk 61/227 done
  Chunk 71/227 done
  Chunk 81/227 done
  Chunk 91/227 done
  Chunk 101/227 done
  Chunk 111/227 done
  Chunk 121/227 done
  Chunk 131/227 done
  Chunk 141/227 done
  Chunk 151/227 done
  Chunk 161/227 done
  Chunk 171/227 done
  Chunk 181/227 done
  Chunk 191/227 done
  Chunk 201/227 done
  Chunk 211/227 done
  Chunk 221/227 done
(38135, 1)
          value_mwh
count  37991.000000
mean  -22616.271049
std    13656.839181
min   -75722.200000
25%   -32283.875000
50%   -21029.500000
75%   -11007.000000
max     -320.500000
                     value_mwh
timestamp                     
2022-01-02 23:00:00  -36595.50
2022-01-03 00:00:00  -36636.75
2022-01-03 01:00:00  -37546.00
2022-01-03 02:00:00  -38352.25
2022-01-03 03:00:00  -38919.75


In [50]:
raw["nuclear"] = test_126

# rebuild frames
frames = {}
for source, source_df in raw.items():
    numeric_cols = source_df.select_dtypes(include="number").columns
    if len(numeric_cols) == 0:
        continue
    frames[source] = source_df[numeric_cols[0]]

df = pd.DataFrame(frames).sort_index().fillna(0)
df = df[df["total_gen"] > 0]

# check uncapped residual with nuclear included
all_known = ["wind_onshore", "wind_offshore", "solar", "biomass",
             "hydro", "hard_coal", "gas", "nuclear", "pumped_storage", "other"]
available_known = [c for c in all_known if c in df.columns]

df["lignite_raw"] = df["total_gen"] - df[available_known].sum(axis=1)
print(df["lignite_raw"].describe())
print(df["lignite_raw"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))
print(f"Negative values: {(df['lignite_raw'] < 0).sum()}")

count     37979.000000
mean      46883.319297
std       16644.433607
min       13615.500000
25%       33664.800000
50%       43989.750000
75%       57741.360000
max      109744.300000
Name: lignite_raw, dtype: float64
0.10    27691.744
0.25    33664.800
0.50    43989.750
0.75    57741.360
0.90    71214.514
Name: lignite_raw, dtype: float64
Negative values: 0


In [51]:
# rebuild without lignite derivation
frames = {}
for source, source_df in raw.items():
    if source in ["other", "pumped_storage"]:
        continue  # exclude unreliable residuals
    numeric_cols = source_df.select_dtypes(include="number").columns
    if len(numeric_cols) == 0:
        continue
    frames[source] = source_df[numeric_cols[0]]

df = pd.DataFrame(frames).sort_index().fillna(0)

# calculate totals from confirmed sources only
renewable_cols = ["wind_onshore", "wind_offshore", "solar", "biomass", "hydro"]
fossil_cols = ["hard_coal", "gas"]

df["renewable_mwh"] = df[renewable_cols].sum(axis=1)
df["fossil_mwh"] = df[fossil_cols].sum(axis=1)
df["total_confirmed"] = df[renewable_cols + fossil_cols].sum(axis=1)

# shares against confirmed generation only
df["renewable_share"] = (df["renewable_mwh"] / df["total_confirmed"]).clip(upper=1.0)
df["fossil_share"] = (df["fossil_mwh"] / df["total_confirmed"]).clip(upper=1.0)

df["hour"] = df.index.hour
df["month"] = df.index.month
df["year"] = df.index.year
df["season"] = df["month"].map({
    12:"winter", 1:"winter", 2:"winter",
    3:"spring",  4:"spring",  5:"spring",
    6:"summer",  7:"summer",  8:"summer",
    9:"autumn",  10:"autumn", 11:"autumn"
})

df = df[df["total_confirmed"] > 0]

print(df.columns.tolist())
print(df.shape)
print(df[["renewable_share", "fossil_share", "renewable_mwh", "fossil_mwh"]].describe())

['wind_onshore', 'wind_offshore', 'solar', 'biomass', 'hydro', 'hard_coal', 'gas', 'total_gen', 'nuclear', 'renewable_mwh', 'fossil_mwh', 'total_confirmed', 'renewable_share', 'fossil_share', 'hour', 'month', 'year', 'season']
(37991, 18)
       renewable_share  fossil_share  renewable_mwh    fossil_mwh
count     37991.000000  37991.000000   37991.000000  37991.000000
mean          0.989471      0.010510   27331.965320    248.755471
std           0.018412      0.018455   10535.772288    113.824585
min           0.000000     -0.098216       0.000000   -405.900000
25%           0.986859      0.006172   19592.145000    186.490000
50%           0.990815      0.009185   25591.750000    219.620000
75%           0.993828      0.013141   34294.375000    282.970000
max           1.000000      1.000000   61603.000000   1070.780000


In [52]:
source_cols = ["wind_onshore", "wind_offshore", "solar", "biomass", 
               "hydro", "hard_coal", "gas"]

for col in source_cols:
    print(f"{col}: mean={df[col].mean():.1f}, max={df[col].max():.1f}")

wind_onshore: mean=9054.6, max=17173.2
wind_offshore: mean=1037.2, max=4117.5
solar: mean=12633.2, max=48499.5
biomass: mean=1670.3, max=3078.2
hydro: mean=2936.6, max=8448.3
hard_coal: mean=126.4, max=225.8
gas: mean=122.4, max=936.3


In [53]:
# look at raw data directly
print("hard_coal raw:")
print(raw["hard_coal"].head(10))
print(raw["hard_coal"].describe())

print("\ngas raw:")
print(raw["gas"].head(10))
print(raw["gas"].describe())

hard_coal raw:
                     value_mwh
timestamp                     
2022-01-02 23:00:00     153.00
2022-01-03 00:00:00     152.00
2022-01-03 01:00:00     153.00
2022-01-03 02:00:00     153.75
2022-01-03 03:00:00     154.75
2022-01-03 04:00:00     154.25
2022-01-03 05:00:00     153.50
2022-01-03 06:00:00     153.00
2022-01-03 07:00:00     149.50
2022-01-03 08:00:00     148.25
          value_mwh
count  37979.000000
mean     126.433412
std       30.229335
min       56.460000
25%      107.000000
50%      117.250000
75%      138.000000
max      225.750000

gas raw:
                     value_mwh
timestamp                     
2022-01-02 23:00:00       0.31
2022-01-03 00:00:00      -0.01
2022-01-03 01:00:00      -0.07
2022-01-03 02:00:00      -1.05
2022-01-03 03:00:00      -1.00
2022-01-03 04:00:00       0.32
2022-01-03 05:00:00      37.55
2022-01-03 06:00:00      88.76
2022-01-03 07:00:00     122.93
2022-01-03 08:00:00     110.17
          value_mwh
count  37991.000000
mean     12

In [54]:
candidates = {
    "hard_coal_a": 4066,
    "hard_coal_b": 1071,
    "hard_coal_c": 4071,
    "gas_a": 1069,
    "gas_b": 4168,
    "gas_c": 1075,
    "lignite_a": 1068,
    "lignite_b": 4065,
}

for name, fid in candidates.items():
    session = make_session()
    url = f"https://www.smard.de/app/chart_data/{fid}/DE/index_hour.json"
    r = session.get(url, headers=HEADERS, timeout=30)
    if r.status_code == 200:
        idx = r.json()
        ts_count = len(idx.get("timestamps", []))
        print(f"{name} (filter {fid}): status 200, {ts_count} timestamps")
    else:
        print(f"{name} (filter {fid}): status {r.status_code}")

hard_coal_a (filter 4066): status 200, 593 timestamps
hard_coal_b (filter 1071): status 404
hard_coal_c (filter 4071): status 200, 593 timestamps
gas_a (filter 1069): status 404
gas_b (filter 4168): status 404
gas_c (filter 1075): status 404
lignite_a (filter 1068): status 404
lignite_b (filter 4065): status 404
